# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Multi-label Classification

Predict multiple labels for each sample.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, accuracy_score

## Understanding Multi-label Data

In [ ]:
# In multi-label classification, each sample can have multiple labels
# The target y is a 2D array: rows=samples, columns=labels

# Example: Movie genres [Action, Comedy, Drama, Horror, Thriller]
# Movie 1: Action + Thriller → [1, 0, 0, 0, 1]
# Movie 2: Comedy + Drama → [0, 1, 1, 0, 0]
# Movie 3: Horror only → [0, 0, 0, 1, 0]

# Create a synthetic movie dataset
np.random.seed(42)
n_samples = 100
n_features = 5
n_labels = 3  # 3 possible genres

# Features: plot characteristics
X = np.random.randn(n_samples, n_features)

# Labels: which genres apply (binary matrix)
y = np.random.randint(0, 2, (n_samples, n_labels))

print(f"X shape: {X.shape} (100 movies, 5 features)")
print(f"y shape: {y.shape} (100 movies, 3 possible labels)")
print(f"\nFirst 5 samples:")
print(f"Labels for first movie: {y[0]} → genres apply: {np.where(y[0])[0]}")
print(f"Labels for second movie: {y[1]} → genres apply: {np.where(y[1])[0]}")
print(f"Labels for third movie: {y[2]} → genres apply: {np.where(y[2])[0]}")

## Step 1: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

## Step 2: Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled.")

## Step 3: Train Multi-label Classifier

Use `MultiOutputClassifier` to train one binary classifier per label.

In [ ]:
# MultiOutputClassifier wraps a base classifier and trains one per output label
# This implements "Binary Relevance" strategy

base_clf = LogisticRegression(max_iter=1000, random_state=42)
clf = MultiOutputClassifier(base_clf)

clf.fit(X_train_scaled, y_train)

print("Multi-label model trained!")
print(f"Number of classifiers: {len(clf.estimators_)}")
print(f"One classifier per label")

## Step 4: Make Predictions

In [ ]:
y_pred = clf.predict(X_test_scaled)

print(f"Predictions shape: {y_pred.shape}")
print(f"\nFirst 5 predictions:")
for i in range(5):
    true_labels = np.where(y_test[i])[0]
    pred_labels = np.where(y_pred[i])[0]
    print(f"Sample {i}: true {true_labels} → predicted {pred_labels}")

## Step 5: Evaluate with Multi-label Metrics

In [ ]:
# Hamming Loss: fraction of labels that are incorrectly predicted
# Lower is better (0 = perfect, 1 = all wrong)
ham_loss = hamming_loss(y_test, y_pred)
print(f"Hamming Loss: {ham_loss:.4f}")
print(f"  → {ham_loss*100:.1f}% of individual label predictions are wrong")

# Jaccard Similarity (Subset Accuracy): 
# Fraction of samples where predicted label set exactly matches true set
# Very strict: any single wrong label counts as total failure
jaccard = accuracy_score(y_test, y_pred)
print(f"\nJaccard Similarity (Subset Accuracy): {jaccard:.4f}")
print(f"  → {jaccard*100:.1f}% of samples have all labels predicted correctly")

## Step 6: Label-level Accuracy

In [ ]:
# Accuracy for each individual label
for label_idx in range(y_test.shape[1]):
    label_accuracy = accuracy_score(y_test[:, label_idx], y_pred[:, label_idx])
    print(f"Label {label_idx} accuracy: {label_accuracy:.4f}")